# 기말 프로젝트 트랙 A: FMB RGB–IR 악천후 세그멘테이션 베이스라인
런타임 > 런타임 유형 변경 > T4 GPU 선택 후 위에서부터 순서대로 실행합니다.

In [ ]:
!nvidia-smi | head -15

In [ ]:
# 1) 코드 준비: 배포받은 vip_trackA_FMB.zip 을 Colab 왼쪽 파일창에 업로드한 뒤 실행
!unzip -q -o vip_trackA_FMB.zip
%cd vip_trackA_FMB
!pip -q install transformers huggingface_hub gdown

In [ ]:
# 2) 데이터 직접 다운로드 (저자 공개 Google Drive, 재배포 없음) -- 실패하면 README 의 '다운로드 실패 시' 참고
!python tools/download_fmb.py --out data/FMB_raw

In [ ]:
# 3) 분할 인덱스 생성 (train 1000 / val 220 / test 280). 출력되는 label values 를 확인!
!python tools/prepare_fmb.py --raw data/FMB_raw --out data/FMB_prepared

In [ ]:
# 4) 비교 기준: RGB 단독
!python train.py --data_root data/FMB_prepared --fusion rgb --epochs 30 --out runs/fmb_rgb --max_minutes 150

In [ ]:
# 5) 베이스라인: RGB+IR 융합(단계별 합산)
!python train.py --data_root data/FMB_prepared --fusion sum --epochs 30 --out runs/fmb_sum --max_minutes 150
# 세션이 끊기면 같은 명령을 다시 실행하면 last.pth 에서 이어서 학습합니다.

In [ ]:
# 6) 제출 파일 생성 (test 280장 예측 PNG -> submission.zip)
!python predict.py --data_root data/FMB_prepared --ckpt runs/fmb_sum/best.pth --split test --out submission
from google.colab import files
files.download('submission.zip')

## 7) 이제 개선해 보세요
`fusion_seg/fusion.py` 에 융합 모듈을 추가하거나(`FUSIONS` 에 등록), `train.py` 의 손실·증강·스케줄을 바꿔 보고,
**RGB 단독 대비 IR 이 기여한 정도**를 val mIoU 표로 정리해 보고서에 넣으세요.